In [13]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import *
import functools
from operator import add

In [14]:
filepath_accidents = 'D:\BDA_0024\ABD_LAB\datasets\Road_accidents.csv'
filepath_time = 'D:\BDA_0024\ABD_LAB\datasets\Road_accidents_time.csv'

In [15]:
spark = SparkSession.builder.appName("Road Accidents Analysis").getOrCreate()

In [16]:
df_accidents = spark.read.csv(filepath_accidents, header=True, inferSchema=True)
df_time = spark.read.csv(filepath_time, header=True, inferSchema=True)

In [17]:
month_cols = ['JANUARY', 'FEBRUARY', 'MARCH', 'APRIL', 'MAY', 'JUNE', 'JULY', 'AUGUST', 'SEPTEMBER', 'OCTOBER', 'NOVEMBER', 'DECEMBER']
time_cols = ['0-3 hrs. (Night)', '3-6 hrs. (Night)', '6-9 hrs (Day)', '9-12 hrs (Day)', '12-15 hrs (Day)', '15-18 hrs (Day)', '18-21 hrs (Night)', '21-24 hrs (Night)']

df_accidents.show(5)

+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------+--------+
|     STATE/UT|YEAR|JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|
+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------+--------+
|A & N Islands|2001|      8|      23|   15|   15| 14|  19|  14|    19|        7|     12|      13|      22|
|A & N Islands|2002|     12|      10|   14|   16| 10|   7|  16|    11|       23|     21|      11|      17|
|A & N Islands|2003|     19|      13|   15|   13| 13|  12|   8|    16|       17|     25|      14|      15|
|A & N Islands|2004|     21|      14|   22|   17| 13|  18|  16|    19|       16|     20|      15|      24|
|A & N Islands|2005|     19|      21|   22|   17| 13|  19|  21|    14|       15|     19|      10|      16|
+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------+--------+
only showing top 5 rows



In [18]:
# 1. Add column 'Total', giving sum of accidents in each month . 

month_sum = functools.reduce(add, [col(m) for m in month_cols])
df_accidents = df_accidents.withColumn('Total', month_sum)
df_accidents.show(5)

+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------+--------+-----+
|     STATE/UT|YEAR|JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|Total|
+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------+--------+-----+
|A & N Islands|2001|      8|      23|   15|   15| 14|  19|  14|    19|        7|     12|      13|      22|  181|
|A & N Islands|2002|     12|      10|   14|   16| 10|   7|  16|    11|       23|     21|      11|      17|  168|
|A & N Islands|2003|     19|      13|   15|   13| 13|  12|   8|    16|       17|     25|      14|      15|  180|
|A & N Islands|2004|     21|      14|   22|   17| 13|  18|  16|    19|       16|     20|      15|      24|  215|
|A & N Islands|2005|     19|      21|   22|   17| 13|  19|  21|    14|       15|     19|      10|      16|  206|
+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------

In [19]:
# 2. Which state has highest number of accidents in year 2013?
df_accidents.filter((col('YEAR') == 2013) & (~col('STATE/UT').isin(['TOTAL', 'Total', 'TOTAL (ALL INDIA)']))) \
    .orderBy(col('Total').desc()) \
    .select('STATE/UT', 'Total') \
    .limit(1) \
    .show(truncate=False)

+----------+-----+
|STATE/UT  |Total|
+----------+-----+
|Tamil Nadu|66238|
+----------+-----+



In [20]:
# 3. Find the average monthly accidents for each state. 
df_accidents.groupBy('STATE/UT') \
    .agg(*[round(avg(m), 2).alias(f'AVG_{m}') for m in month_cols]) \
    .show(truncate=False)

+-----------------+-----------+------------+---------+---------+-------+--------+--------+----------+-------------+-----------+------------+------------+
|STATE/UT         |AVG_JANUARY|AVG_FEBRUARY|AVG_MARCH|AVG_APRIL|AVG_MAY|AVG_JUNE|AVG_JULY|AVG_AUGUST|AVG_SEPTEMBER|AVG_OCTOBER|AVG_NOVEMBER|AVG_DECEMBER|
+-----------------+-----------+------------+---------+---------+-------+--------+--------+----------+-------------+-----------+------------+------------+
|Nagaland         |5.07       |5.86        |5.14     |4.93     |4.0    |5.36    |5.07    |3.64      |3.07         |5.21       |3.29        |5.14        |
|Karnataka        |3573.64    |3407.5      |3587.43  |3580.71  |3929.5 |3482.14 |3269.14 |3236.14   |3222.0       |3349.29    |3406.43     |3724.71     |
|Odisha           |751.21     |685.07      |745.64   |693.0    |773.5  |696.79  |641.93  |591.36    |591.71       |627.86     |678.64      |754.5       |
|Kerala           |3361.14    |3058.0      |3182.07  |3087.43  |3227.0 |2926

In [21]:
# 4. Which month has highest accidents?
df_accidents.select([sum(m).alias(m) for m in month_cols]).show()

+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
|JANUARY|FEBRUARY| MARCH| APRIL|   MAY|  JUNE|  JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
| 482719|  459272|486141|479663|521563|473574|440263|438351|   435302| 454961|  457192|  473905|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+



In [22]:
# 5. Which time slot (like 0-3, 3-6 etc) has more accidents? What about 'Karnataka' state? 
print("Overall total accidents per time slot:")
df_time.select([sum(col(f'`{t}`')).alias(t) for t in time_cols]).show(truncate=False)

print("Karnataka total accidents per time slot:")
df_time.filter(col('STATE/UT') == 'Karnataka') \
    .select([sum(col(f'`{t}`')).alias(t) for t in time_cols]) \
    .show(truncate=False)

Overall total accidents per time slot:
+----------------+----------------+-------------+--------------+---------------+---------------+-----------------+-----------------+
|0-3 hrs. (Night)|3-6 hrs. (Night)|6-9 hrs (Day)|9-12 hrs (Day)|12-15 hrs (Day)|15-18 hrs (Day)|18-21 hrs (Night)|21-24 hrs (Night)|
+----------------+----------------+-------------+--------------+---------------+---------------+-----------------+-----------------+
|390197          |474926          |671864       |859444        |824089         |906639         |873630           |602117           |
+----------------+----------------+-------------+--------------+---------------+---------------+-----------------+-----------------+

Karnataka total accidents per time slot:
+----------------+----------------+-------------+--------------+---------------+---------------+-----------------+-----------------+
|0-3 hrs. (Night)|3-6 hrs. (Night)|6-9 hrs (Day)|9-12 hrs (Day)|12-15 hrs (Day)|15-18 hrs (Day)|18-21 hrs (Night)|21-24 h

In [23]:
# 6. Which state has more accidents from year 2001 to 2014?
df_accidents.filter(~col('STATE/UT').isin(['TOTAL', 'Total', 'TOTAL (ALL INDIA)'])) \
    .groupBy('STATE/UT') \
    .agg(sum('Total').alias('GRAND_TOTAL')) \
    .orderBy(col('GRAND_TOTAL').desc()) \
    .limit(1) \
    .show(truncate=False)

+----------+-----------+
|STATE/UT  |GRAND_TOTAL|
+----------+-----------+
|Tamil Nadu|852073     |
+----------+-----------+



In [24]:
# 7. List states whose accidents number in year 2014 is less than state average from 2001 to 2014.
state_avg = df_accidents.groupBy('STATE/UT').agg(avg('Total').alias('STATE_AVG'))

df_2014 = df_accidents.filter(col('YEAR') == 2014).select('STATE/UT', col('Total').alias('TOTAL_2014'))

df_2014.join(state_avg, 'STATE/UT') \
    .filter(col('TOTAL_2014') < col('STATE_AVG')) \
    .select('STATE/UT', 'TOTAL_2014', round('STATE_AVG', 2).alias('STATE_AVG')) \
    .show(100, truncate=False)

+-----------------+----------+---------+
|STATE/UT         |TOTAL_2014|STATE_AVG|
+-----------------+----------+---------+
|Nagaland         |28        |55.79    |
|Kerala           |35872     |37011.5  |
|Daman & Diu      |39        |45.93    |
|Jammu & Kashmir  |5778      |5915.71  |
|Puducherry       |671       |1513.5   |
|Arunachal Pradesh|185       |242.07   |
|Sikkim           |130       |189.0    |
|Chandigarh       |366       |457.93   |
|Maharashtra      |44382     |45829.57 |
|Tripura          |716       |752.43   |
|Uttarakhand      |801       |1299.43  |
+-----------------+----------+---------+



In [25]:
# 8. Which month has more accidents over the year?
monthly_totals = df_accidents.select([sum(col(m)).alias(m) for m in month_cols])
monthly_totals.show()

unpivot_expr = "stack(12, " + ", ".join([f"'{m}', `{m}`" for m in month_cols]) + ") as (Month, Total_Accidents)"

monthly_totals.selectExpr(unpivot_expr) \
    .orderBy(col('Total_Accidents').desc()) \
    .limit(1) \
    .show()

+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
|JANUARY|FEBRUARY| MARCH| APRIL|   MAY|  JUNE|  JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
| 482719|  459272|486141|479663|521563|473574|440263|438351|   435302| 454961|  457192|  473905|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+

+-----+---------------+
|Month|Total_Accidents|
+-----+---------------+
|  MAY|         521563|
+-----+---------------+

